In [ ]:
# Install dependencies as needed:
#Kaggle API key: KGAT_d585e542ab04edffd1d9e70b83ad632c
%pip install kagglehub[pandas-datasets]
%pip install pyspark
import kagglehub
kagglehub.login()
from kagglehub import KaggleDatasetAdapter

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
file_path = "cardio_train.csv"
# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "sulianova/cardiovascular-disease-dataset",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
  pandas_kwargs={"sep": ";"}
)

print("First 5 records:", df.head())
print(df.shape)

First 5 records:    id    age  gender  height  weight  ap_hi  ap_lo  cholesterol  gluc  smoke  \
0   0  18393       2     168    62.0    110     80            1     1      0   
1   1  20228       1     156    85.0    140     90            3     1      0   
2   2  18857       1     165    64.0    130     70            3     1      0   
3   3  17623       2     169    82.0    150    100            1     1      0   
4   4  17474       1     156    56.0    100     60            1     1      0   

   alco  active  cardio  
0     0       1       0  
1     0       1       1  
2     0       0       1  
3     0       1       1  
4     0       0       0  
(70000, 13)


In [36]:
%pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [47]:
from ucimlrepo import fetch_ucirepo
#fetch dataset

heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features
Y = heart_disease.data.targets

#print(heart_disease.metadata)

#print(heart_disease.variables)
print(heart_disease.data.targets)

     num
0      0
1      2
2      1
3      0
4      0
..   ...
298    1
299    2
300    3
301    1
302    0

[303 rows x 1 columns]


In [49]:
import pandas as pd

folder = r"C:\Users\prest\Downloads"
files = ["DEMO_J", "BPX_J", "BMX_J", "GLU_J", "GHB_J", "TCHOL_J"]

dfs = [pd.read_sas(rf"{folder}\{f}.XPT", format = "xport") for f in files]

df = dfs[0]
for d in dfs[1:]:
    df = df.merge(d, on="SEQN", how="left")
print(df.shape)
print(df.head)

(9254, 92)
<bound method NDFrame.head of           SEQN  SDDSRVYR  RIDSTATR  RIAGENDR  RIDAGEYR  RIDAGEMN  RIDRETH1  \
0      93703.0      10.0       2.0       2.0       2.0       NaN       5.0   
1      93704.0      10.0       2.0       1.0       2.0       NaN       3.0   
2      93705.0      10.0       2.0       2.0      66.0       NaN       4.0   
3      93706.0      10.0       2.0       1.0      18.0       NaN       5.0   
4      93707.0      10.0       2.0       1.0      13.0       NaN       5.0   
...        ...       ...       ...       ...       ...       ...       ...   
9249  102952.0      10.0       2.0       2.0      70.0       NaN       5.0   
9250  102953.0      10.0       2.0       1.0      42.0       NaN       1.0   
9251  102954.0      10.0       2.0       2.0      41.0       NaN       4.0   
9252  102955.0      10.0       2.0       2.0      14.0       NaN       4.0   
9253  102956.0      10.0       2.0       1.0      38.0       NaN       3.0   

      RIDRETH3  RIDEXM

In [50]:
cols = {
    "SEQN": "id", "RIDAGEYR": "age", "RIAGENDR": "sex",
    "BPXSY1": "sys_bp", "BPXDI1": "dia_bp", "BMXBMI": "bmi",
    "LBXGLU": "glucose", "LBXGH": "hba1c", "LBXTC": "cholesterol",
}

nhanes = df[list(cols)].rename(columns=cols)
nhanes = nhanes[nhanes["age"] >= 18].copy()
nhanes["id"] = nhanes["id"].astype(int)
nhanes["source"] = "NHANES"

# label: diabetes by standard lab cutoffs
has_test = nhanes["hba1c"].notna() | nhanes["glucose"].notna()
nhanes = nhanes[has_test].copy()
nhanes["diabetes"] = ((nhanes["hba1c"] >= 6.5) | (nhanes["glucose"] >= 126)).astype(int)
nhanes = nhanes.reset_index(drop=True)

print(nhanes.shape)
print(nhanes["diabetes"].value_counts(normalize=True).round(3))
nhanes.to_csv("nhanes_clean.csv", index=False)

(5264, 11)
diabetes
0    0.837
1    0.163
Name: proportion, dtype: float64


In [51]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, recall_score, precision_score

def rule_score(r):
    s = 0
    # age
    if r["age"] >= 65: s += 3
    elif r["age"] >= 45: s += 2
    # BMI
    if r["bmi"] >= 30: s += 3
    elif r["bmi"] >= 25: s += 1
    # blood pressure (hypertension thresholds)
    if (r["sys_bp"] >= 140) or (r["dia_bp"] >= 90): s += 2
    elif (r["sys_bp"] >= 130) or (r["dia_bp"] >= 80): s += 1
    # cholesterol
    if r["cholesterol"] >= 240: s += 2
    elif r["cholesterol"] >= 200: s += 1
    return s

nhanes["rule_score"] = nhanes.apply(rule_score, axis=1)

# one split, reused for every model so comparisons are fair
train, test = train_test_split(
    nhanes, test_size=0.2, stratify=nhanes["diabetes"], random_state=42
)

auc = roc_auc_score(test["diabetes"], test["rule_score"])
print("Rule-based AUC:", round(auc, 3))

for cutoff in [3, 4, 5]:
    pred = (test["rule_score"] >= cutoff).astype(int)
    print(cutoff,
          "recall:", round(recall_score(test["diabetes"], pred), 3),
          "precision:", round(precision_score(test["diabetes"], pred), 3))

Rule-based AUC: 0.71
3 recall: 0.977 precision: 0.213
4 recall: 0.808 precision: 0.231
5 recall: 0.669 precision: 0.256
